In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import label_binarize

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from matplotlib.backends.backend_pdf import PdfPages

# ==== CONFIGURACIÓN ====
CATEGORIA = "Standard_FE_ALL"
INTENTO = 1
BASE_INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected"
BASE_OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\models"
fecha_actual = pd.Timestamp.today().strftime("%Y-%m-%d")
OUTPUT_PATH = os.path.join(BASE_OUTPUT_PATH, f"MLP_KERAS_SMOTE_{INTENTO}_{CATEGORIA}_{fecha_actual}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ==== CARGA DE DATOS ====
X_train = pd.read_parquet(os.path.join(BASE_INPUT_PATH, f"X_train_{CATEGORIA}.parquet"))
X_test = pd.read_parquet(os.path.join(BASE_INPUT_PATH, f"X_test_{CATEGORIA}.parquet"))
y_train = pd.read_parquet(os.path.join(BASE_INPUT_PATH, f"y_train_{CATEGORIA}.parquet")).squeeze()
y_test = pd.read_parquet(os.path.join(BASE_INPUT_PATH, f"y_test_{CATEGORIA}.parquet")).squeeze()

# ==== SMOTE ====
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

# ==== FOCAL LOSS ====
def focal_loss(gamma=2., alpha=.25):
    def focal_loss_fixed(y_true, y_pred):
        epsilon = K.epsilon()
        y_pred = K.clip(y_pred, epsilon, 1. - epsilon)
        cross_entropy = -y_true * K.log(y_pred)
        weight = alpha * K.pow(1 - y_pred, gamma)
        loss = weight * cross_entropy
        return K.mean(K.sum(loss, axis=1))
    return focal_loss_fixed

# ==== PREPROCESAMIENTO ====
X_train_res = StandardScaler().fit_transform(X_train_res)
X_test = StandardScaler().fit(X_train).transform(X_test)

y_train_res = y_train_res.astype(int)
y_test = y_test.astype(int)

num_classes = np.max([y_train_res.max(), y_test.max()]) + 1
y_train_cat = to_categorical(y_train_res, num_classes)
y_test_cat = to_categorical(y_test, num_classes)
classes = np.arange(num_classes)

# ==== MODELO ====
model = Sequential([
    Dense(128, input_dim=X_train_res.shape[1], activation='relu'),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(num_classes, activation='softmax')
])

model.compile(loss=focal_loss(gamma=2, alpha=0.25),
              optimizer='adam',
              metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(X_train_res, y_train_cat,
                    validation_data=(X_test, y_test_cat),
                    epochs=100,
                    batch_size=64,
                    callbacks=[early_stop],
                    verbose=1)

# ==== PREDICCIONES ====
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# ==== REPORTES ====
report_test = pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).T
report_test.to_csv(os.path.join(OUTPUT_PATH, f"metricas_por_clase.csv"))

# ==== PDF DE GRÁFICOS ====
with PdfPages(os.path.join(OUTPUT_PATH, "reporte_metricas.pdf")) as pdf:
    # Matriz de confusión
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues')
    plt.title("Matriz de Confusión")
    pdf.savefig(); plt.close()

    # ROC Curves
    y_bin = label_binarize(y_test, classes=classes)
    for i, clase in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_probs[:, i])
        plt.plot(fpr, tpr, label=f"Clase {clase} (AUC={auc(fpr, tpr):.2f})")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title("Curvas ROC"); plt.legend(); pdf.savefig(); plt.close()

    # PR Curves
    for i, clase in enumerate(classes):
        precision, recall, _ = precision_recall_curve(y_bin[:, i], y_pred_probs[:, i])
        plt.plot(recall, precision, label=f"Clase {clase}")
    plt.title("Curvas Precision-Recall"); plt.legend(); pdf.savefig(); plt.close()

# ==== RESUMEN GLOBAL ====
resumen = {
    "modelo": CATEGORIA,
    "accuracy": accuracy_score(y_test, y_pred),
    "macro_f1": report_test.loc["macro avg", "f1-score"],
    "weighted_f1": report_test.loc["weighted avg", "f1-score"]
}
pd.DataFrame([resumen]).to_csv(os.path.join(OUTPUT_PATH, "resumen_modelo.csv"), index=False)

print("\n✅ Modelo entrenado y resultados guardados.")


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
9716/9716 ━━━━━━━━━━━━━━━━━━━━ 35s 3ms/step - accuracy: 0.4076 - loss: 0.1962 - val_accuracy: 0.5044 - val_loss: 0.1626
Epoch 2/100
9716/9716 ━━━━━━━━━━━━━━━━━━━━ 32s 3ms/step - accuracy: 0.4423 - loss: 0.1695 - val_accuracy: 0.4792 - val_loss: 0.1649
Epoch 3/100
9716/9716 ━━━━━━━━━━━━━━━━━━━━ 33s 3ms/step - accuracy: 0.4461 - loss: 0.1673 - val_accuracy: 0.4641 - val_loss: 0.1664
Epoch 4/100
9716/9716 ━━━━━━━━━━━━━━━━━━━━ 45s 5ms/step - accuracy: 0.4456 - loss: 0.1663 - val_accuracy: 0.4570 - val_loss: 0.1617
Epoch 5/100
9716/9716 ━━━━━━━━━━━━━━━━━━━━ 35s 4ms/step - accuracy: 0.4482 - loss: 0.1651 - val_accuracy: 0.4891 - val_loss: 0.1653
Epoch 6/100
9716/9716 ━━━━━━━━━━━━━━━━━━━━ 41s 4ms/step - accuracy: 0.4488 - loss: 0.1643 - val_accuracy: 0.4723 - val_loss: 0.1643
Epoch 7/100
9716/9716 ━━━━━━━━━━━━━━━━━━━━ 41s 4ms/step - accuracy: 0.4513 - loss: 0.1638 - val_accuracy: 0.5021 - val_loss: 0.1616
Epoch 8/100
9716/9716 ━━━━━━━━━━━━━━━━━━━━ 42s 4ms/step - accuracy: 0.4529 -

c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1188: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(



✅ Modelo entrenado y resultados guardados.
